04_prepare_dashboard_data.py: подготовка dashboard_points_v3.parquet

Автономный скрипт. Объединяет точечные данные:

1. `model_scores_points_v3.parquet` (62.6M, scores + DQ aggregate + risk).
2. `annotated_v3.parquet` (149M, физика + energy + DQ subflags), фильтруем по calval и test flight_ids.

Выход: `dashboard_points_v3.parquet` (~3 GB), pre-joined point-level data для Streamlit/Dash дашборда.

## Замечания по реализации

`annotated_v3` уже содержит все 33 enriched-колонки (energy_ratio, energy_deviation) поверх raw telemetry, поэтому отдельно `enriched_v4.parquet` не читаем. Один проход вместо двух, меньше I/O, нет риска row-order mismatch.

Вместо huge Python sets из 62.6M row_ids используем `pd.merge(..., validate='one_to_one')` плюс technical-колонку `has_extra` для контроля покрытия. Экономит порядка 3-4 GB пиковой памяти.

После merge проверяем схему набором требуемых колонок, чтобы не было silent filter output_cols.

DQ subflags (dt_sec, gap_flag, dq_derivative_bad, phase_inconsistent, energy_deviation_extreme) попадают в дашборд: нужны, чтобы объяснить почему dq_soft=1 ("because derivative_bad=1").

Колонки `split` и `risk_level` сохраняются как categorical для экономии памяти.

Старый output удаляется перед записью, на случай неполной записи на Drive.

Sanity-flight выбирается динамически по max risk_score, без хардкода id.

## Время

~10-15 минут (Drive I/O bound, один проход через annotated_v3).

## Память

Пик около 7 GB (scores + extra DataFrame в RAM перед merge).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import os
import gc
import json
import time

DATA_DIR = '/content/drive/MyDrive/thesis_processed'
ARTIFACTS_DIR = os.path.join(DATA_DIR, 'models_v3_artifacts')

# Inputs (теперь только 2 файла, без enriched_v4)
MODEL_SCORES_PATH = os.path.join(ARTIFACTS_DIR, 'model_scores_points_v3.parquet')
INPUT_ANNOTATED = os.path.join(DATA_DIR, 'european_flights_annotated_v3.parquet')
SPLIT_PATH = os.path.join(ARTIFACTS_DIR, 'split_metadata.json')

# Output
DASHBOARD_POINTS_PATH = os.path.join(ARTIFACTS_DIR, 'dashboard_points_v3.parquet')

# Phase name mapping
PHASE_NAMES = {-1: 'unknown', 0: 'ground', 1: 'takeoff', 2: 'climb',
               3: 'cruise', 4: 'descent', 5: 'approach', 6: 'landing'}

# Требуемый output schema (assert all present after merge)
REQUIRED_OUTPUT_COLS = [
    # Identity
    'row_id', 'flight_id', 'timestamp', 'flight_phase', 'phase_name', 'split',

    # Geo
    'latitude', 'longitude',

    # Physics
    'altitude', 'groundspeed', 'vertical_rate',

    # Energy (из enriched_v4, теперь в annotated_v3)
    'energy_ratio', 'energy_deviation',

    # DQ aggregate flags (3-tier)
    'dq_hard_flag', 'dq_soft_flag', 'feature_quality_flag', 'is_clean_reference',

    # DQ subflags (для объяснимости в дашборде)
    'dt_sec', 'gap_flag', 'dq_derivative_bad',
    'phase_inconsistent', 'energy_deviation_extreme',

    # Model scores (phase-aware percentiles)
    'if_score_phase_pct', 'hdb_score_phase_pct',
    'lstm_score_max_phase_pct', 'lstm_score_count',

    # Ensemble
    'ensemble_score', 'ensemble_count',

    # Final risk (calibrated to clean calval ECDF)
    'risk_score', 'risk_level',
]

print('Inputs:')
for path in [MODEL_SCORES_PATH, INPUT_ANNOTATED, SPLIT_PATH]:
    exists = os.path.exists(path)
    size_mb = os.path.getsize(path) / 1e6 if exists else 0
    print(f'  {os.path.basename(path):<45s}: {exists}, {size_mb:>8.1f} MB')

print(f'\nOutput: {os.path.basename(DASHBOARD_POINTS_PATH)}')

Mounted at /content/drive
Inputs:
  model_scores_points_v3.parquet               : True,   2377.9 MB
  european_flights_annotated_v3.parquet        : True,   8357.4 MB
  split_metadata.json                          : True,      0.3 MB

Output: dashboard_points_v3.parquet


## 1. Load split metadata

In [2]:
print('\nStep 1: Load split metadata')

with open(SPLIT_PATH) as f:
    split_meta = json.load(f)

calval_ids = np.array(sorted(split_meta['calibration_val_flights']), dtype=np.int64)
test_ids = np.array(sorted(split_meta['test_flights']), dtype=np.int64)
relevant_set = set(calval_ids.tolist()) | set(test_ids.tolist())

print(f'  calval flights: {len(calval_ids):,}')
print(f'  test flights:   {len(test_ids):,}')
print(f'  Total relevant: {len(relevant_set):,}')


=== Step 1: Load split metadata ===
  calval flights: 4,307
  test flights:   8,251
  Total relevant: 12,558


## 2. Verify annotated_v3 has required columns

In [3]:
print('\nStep 2: Verify annotated_v3 schema')

pf_anno_schema = pq.ParquetFile(INPUT_ANNOTATED)
all_anno_cols = pf_anno_schema.schema_arrow.names
print(f'  Total columns in annotated_v3: {len(all_anno_cols)}')

# Требуемые физика + energy + DQ subflags колонки
required_anno_cols = [
    'flight_id',
    'latitude', 'longitude',
    'altitude', 'groundspeed', 'vertical_rate',
    'energy_ratio', 'energy_deviation',
    'dt_sec', 'gap_flag',
    'dq_derivative_bad', 'phase_inconsistent', 'energy_deviation_extreme',
]

missing_anno = [c for c in required_anno_cols if c not in all_anno_cols]

# Некоторые колонки могут иметь slight name variations, поищем alternatives
if missing_anno:
    print(f'  WARN: missing in annotated_v3: {missing_anno}')
    print(f'  Searching for alternative names...')

    name_alternatives = {
        'energy_deviation': ['energy_deviation_from_phase_norm', 'energy_dev_phase'],
        'dq_derivative_bad': ['derivative_bad', 'dq_derivative'],
        'phase_inconsistent': ['phase_inconsistency_flag'],
        'energy_deviation_extreme': ['energy_extreme_flag', 'energy_outlier_flag'],
    }

    rename_map = {}
    for missing_col in missing_anno[:]:  # copy list для safe mutation
        if missing_col in name_alternatives:
            for alt in name_alternatives[missing_col]:
                if alt in all_anno_cols:
                    rename_map[alt] = missing_col
                    missing_anno.remove(missing_col)
                    print(f'    {missing_col}: use {alt}')
                    break

    if missing_anno:
        # Завершаем с ошибкой, если что-то реально отсутствует
        print(f'\nERROR: still missing in annotated_v3: {missing_anno}')
        print(f'Available columns: {all_anno_cols}')
        raise AssertionError(f'Missing required columns: {missing_anno}')
else:
    rename_map = {}
    print(f'  All required columns present')

# Adjust read list to use alternative names
effective_anno_cols = list(required_anno_cols)
for desired, alt in rename_map.items():
    # rename_map: alt в desired, читаем alt, потом rename в desired
    pass
# Reset: read what's actually there, rename after
actual_read_cols = ['flight_id'] + [
    next((alt for alt, desired in rename_map.items() if desired == c), c)
    for c in required_anno_cols[1:]  # skip flight_id
]
print(f'  Will read columns: {actual_read_cols}')
print(f'  Will rename: {rename_map}')


=== Step 2: Verify annotated_v3 schema ===
  Total columns in annotated_v3: 41
  All required columns present ✓
  Will read columns: ['flight_id', 'latitude', 'longitude', 'altitude', 'groundspeed', 'vertical_rate', 'energy_ratio', 'energy_deviation', 'dt_sec', 'gap_flag', 'dq_derivative_bad', 'phase_inconsistent', 'energy_deviation_extreme']
  Will rename: {}


## 3. Load model_scores_points_v3 fully into RAM

In [4]:
print('\nStep 3: Load model_scores_points_v3')
t0 = time.time()

model_score_cols = [
    'row_id', 'flight_id', 'timestamp', 'flight_phase', 'split',
    'dq_hard_flag', 'dq_soft_flag', 'feature_quality_flag', 'is_clean_reference',
    'if_score_phase_pct', 'hdb_score_phase_pct',
    'lstm_score_max_phase_pct', 'lstm_score_count',
    'ensemble_score', 'ensemble_count',
    'risk_score', 'risk_level',
]

scores_df = pq.read_table(MODEL_SCORES_PATH, columns=model_score_cols).to_pandas()
print(f'  Loaded: {len(scores_df):,} rows in {time.time() - t0:.0f}s')
print(f'  Memory before opt: {scores_df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

# Оптимизация dtype
scores_df['flight_phase'] = scores_df['flight_phase'].astype('int8')
for col in ['dq_hard_flag', 'dq_soft_flag', 'feature_quality_flag',
            'is_clean_reference', 'ensemble_count']:
    scores_df[col] = scores_df[col].astype('uint8')
scores_df['lstm_score_count'] = scores_df['lstm_score_count'].astype('int16')
for col in ['if_score_phase_pct', 'hdb_score_phase_pct',
            'lstm_score_max_phase_pct', 'ensemble_score', 'risk_score']:
    scores_df[col] = scores_df[col].astype('float32')

# Categorical
scores_df['split'] = scores_df['split'].astype('category')
scores_df['risk_level'] = scores_df['risk_level'].astype('category')

print(f'  Memory after opt:  {scores_df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

# Sanity
assert scores_df['row_id'].is_unique
print(f'  row_id unique')

n_scores = len(scores_df)


=== Step 3: Load model_scores_points_v3 ===
  Loaded: 62,578,761 rows in 38s
  Memory before opt: 3.38 GB
  Memory after opt:  3.38 GB
  row_id unique ✓


## 4. Streaming через annotated_v3, извлекаем ВСЕ нужные колонки за один проход

In [5]:
print('\nStep 4: Stream annotated_v3 (one pass, all columns)')
t0 = time.time()

pf_anno = pq.ParquetFile(INPUT_ANNOTATED)
extra_chunks = []
global_offset = 0
n_batches = 0
n_kept_total = 0

for batch in pf_anno.iter_batches(batch_size=5_000_000, columns=actual_read_cols):
    df = batch.to_pandas()
    n = len(df)
    df['row_id'] = np.arange(global_offset, global_offset + n, dtype=np.int64)
    global_offset += n
    n_batches += 1

    # Применяем rename map при необходимости (чтобы output names совпадали с REQUIRED_OUTPUT_COLS)
    if rename_map:
        df = df.rename(columns=rename_map)

    # Фильтр по relevant flights
    is_relevant = df['flight_id'].isin(relevant_set)
    keep_cols = [c for c in required_anno_cols if c != 'flight_id'] + ['row_id']
    df_filt = df.loc[is_relevant, keep_cols].copy()

    # Оптимизация dtype
    for col in ['latitude', 'longitude', 'altitude', 'groundspeed',
                'vertical_rate', 'energy_ratio', 'energy_deviation', 'dt_sec']:
        df_filt[col] = df_filt[col].astype('float32')

    for col in ['gap_flag', 'dq_derivative_bad',
                'phase_inconsistent', 'energy_deviation_extreme']:
        df_filt[col] = df_filt[col].astype('uint8')

    extra_chunks.append(df_filt)
    n_kept_total += len(df_filt)

    print(f'  batch {n_batches}: rows={n:,}, kept={len(df_filt):,}, '
          f'total_kept={n_kept_total:,}', end='\r', flush=True)

    del df, df_filt

print()
print(f'  Streaming done: {time.time() - t0:.0f}s, {n_batches} batches')

# Проверка, что общий счётчик строк совпадает с annotated
n_annotated = pq.ParquetFile(INPUT_ANNOTATED).metadata.num_rows
assert global_offset == n_annotated, (
    f'global_offset {global_offset} != annotated rows {n_annotated}'
)
print(f'  Total annotated rows processed: {n_annotated:,}')

extra_df = pd.concat(extra_chunks, ignore_index=True)
del extra_chunks
print(f'  Extra DataFrame: {len(extra_df):,} rows, '
      f'{extra_df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

# Добавляем technical marker для coverage assert после merge
extra_df['has_extra'] = np.uint8(1)


=== Step 4: Stream annotated_v3 (one pass, all columns) ===

  Streaming done: 106s, 30 batches
  Total annotated rows processed: 149,129,454 ✓
  Extra DataFrame: 63,019,803 rows, 2.77 GB


## 5. Объединение scores с extra и проверка покрытия

Используем pd.merge с validate='one_to_one', он сам проверяет уникальность row_id в обеих таблицах. После merge проверяем технической колонкой, что все scored rows получили extra-колонки.

In [6]:
print('\nStep 5: Merge scores with extra (with validation)')
t0 = time.time()

merged = scores_df.merge(extra_df, on='row_id', how='left', validate='one_to_one')
del scores_df, extra_df
print(f'  After merge: {len(merged):,} rows, '
      f'{merged.memory_usage(deep=True).sum() / 1e9:.2f} GB')

# Проверяем покрытие через has_extra
n_missing = int(merged['has_extra'].fillna(0).eq(0).sum())
if n_missing > 0:
    print(f'\nERROR: {n_missing:,} scored rows missing physics/energy data!')
    # Find examples
    missing_examples = merged.loc[
        merged['has_extra'].fillna(0).eq(0), ['row_id', 'flight_id']
    ].head(5)
    print('Examples:')
    print(missing_examples)
    raise AssertionError(f'{n_missing} scored rows missing in extra DataFrame')

print(f'  Coverage check passed: 100% scored rows have physics/energy')
merged = merged.drop(columns=['has_extra'])

# Sanity row_id ещё уникален
assert merged['row_id'].is_unique
print(f'  row_id unique after merge')

print(f'\nMerge time: {time.time() - t0:.0f}s')

# Coverage report
print(f'\nNon-NaN coverage (sanity):')
for col in ['latitude', 'altitude', 'energy_ratio', 'energy_deviation',
            'gap_flag', 'dq_derivative_bad', 'phase_inconsistent',
            'energy_deviation_extreme']:
    n_cov = merged[col].notna().sum()
    pct = n_cov / len(merged) * 100
    print(f'  {col:>20s}: {n_cov:,} ({pct:>6.2f}%)')


=== Step 5: Merge scores ← extra (with validation) ===
  After merge: 62,578,761 rows, 5.69 GB
  Coverage check passed: 100% scored rows have physics/energy ✓
  row_id unique after merge ✓

Merge time: 166s

Non-NaN coverage (sanity):
                        latitude:   62,578,761 (100.00%)
                        altitude:   62,578,761 (100.00%)
                    energy_ratio:   62,578,761 (100.00%)
                energy_deviation:   62,578,761 (100.00%)
                        gap_flag:   62,578,761 (100.00%)
               dq_derivative_bad:   62,578,761 (100.00%)
              phase_inconsistent:   62,578,761 (100.00%)
        energy_deviation_extreme:   62,578,761 (100.00%)


## 6. Add phase_name (denormalized для дашборда)

In [7]:
print('\nStep 6: Add phase_name + categorical types')

merged['phase_name'] = merged['flight_phase'].map(PHASE_NAMES).astype('category')

phase_dist = merged['phase_name'].value_counts()
print(f'  Phase distribution:')
for phase, n in phase_dist.items():
    print(f'    {phase:>10s}: {n:,}')


=== Step 6: Add phase_name + categorical types ===
  Phase distribution:
        cruise:   31,961,435
         climb:   11,850,752
       descent:   11,036,050
      approach:    6,554,313
       landing:      868,204
       takeoff:      165,456
        ground:      142,551


## 7. Проверка набора колонок

In [8]:
print('\nStep 7: проверка набора колонок')

missing_cols = [c for c in REQUIRED_OUTPUT_COLS if c not in merged.columns]
if missing_cols:
    print(f'\nERROR: missing required output columns: {missing_cols}')
    print(f'Available columns: {sorted(merged.columns.tolist())}')
    raise AssertionError(f'Missing required dashboard columns: {missing_cols}')

print(f'  All {len(REQUIRED_OUTPUT_COLS)} required columns present')


=== Step 7: Strict schema check ===
  All 30 required columns present ✓


## 8. Sort by (flight_id, timestamp)

In [9]:
print('\nStep 8: Sort by (flight_id, timestamp)')
t0 = time.time()

merged = merged.sort_values(['flight_id', 'timestamp']).reset_index(drop=True)
print(f'  Sort done: {time.time() - t0:.0f}s')


=== Step 8: Sort by (flight_id, timestamp) ===
  Sort done: 14s


## 9. Запись dashboard_points_v3.parquet

Старый файл удаляем заранее, на случай неполной записи на Drive.

In [11]:
print('\nStep 9: Write dashboard_points_v3.parquet')

# Удаляем старый output если есть
if os.path.exists(DASHBOARD_POINTS_PATH):
    print(f'  Removing existing file...')
    os.remove(DASHBOARD_POINTS_PATH)

t0 = time.time()
print(f'  Writing {len(merged):,} rows x {len(REQUIRED_OUTPUT_COLS)} cols...')

chunk_size = 5_000_000
writer = None
for cstart in range(0, len(merged), chunk_size):
    cend = min(cstart + chunk_size, len(merged))
    chunk = merged.iloc[cstart:cend][REQUIRED_OUTPUT_COLS].copy()
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(
            DASHBOARD_POINTS_PATH, table.schema,
            compression='snappy'
        )
    # row_group_size=10K даёт ~6.3K row_groups, pyarrow пропускает irrelevant
    # ones при filtering by flight_id
    writer.write_table(table, row_group_size=10_000)
    print(f'  Written {cend:,} / {len(merged):,}', end='\r', flush=True)
    del chunk, table

if writer is not None:
    writer.close()

print()
size_gb = os.path.getsize(DASHBOARD_POINTS_PATH) / 1e9
print(f'\n  Saved: {DASHBOARD_POINTS_PATH}')
print(f'  Size:  {size_gb:.2f} GB')
print(f'  Time:  {time.time() - t0:.0f}s')


=== Step 9: Write dashboard_points_v3.parquet ===
  Writing 62,578,761 rows × 30 cols...


  Saved: /content/drive/MyDrive/thesis_processed/models_v3_artifacts/dashboard_points_v3.parquet
  Size:  3.16 GB
  Time:  191s


## 10. Dynamic sanity check (highest-risk flight)

In [12]:
print('\nStep 10: Sanity check (read highest-risk flight via filter)')

# Выбираем рейс динамически по максимальному risk_score
top_idx = merged['risk_score'].idxmax()
test_flight_id = int(merged.loc[top_idx, 'flight_id'])
top_risk = float(merged.loc[top_idx, 'risk_score'])
print(f'  Picked flight {test_flight_id} (max risk_score={top_risk:.2f})')

del merged

t0 = time.time()
test_data = pq.read_table(
    DASHBOARD_POINTS_PATH,
    filters=[('flight_id', '=', test_flight_id)]
).to_pandas()
read_time = time.time() - t0

print(f'  Read time: {read_time:.2f}s')
print(f'  Points: {len(test_data):,}')
print(f'  Columns ({len(test_data.columns)}): {test_data.columns.tolist()}')
print(f'  Risk score range: [{test_data["risk_score"].min():.2f}, '
      f'{test_data["risk_score"].max():.2f}]')
print(f'  Risk level distribution:')
for level, n in test_data['risk_level'].value_counts().items():
    print(f'    {level}: {n:,}')
print(f'  Phase distribution:')
for phase, n in test_data['phase_name'].value_counts().items():
    print(f'    {phase}: {n:,}')

# Sanity: проверим что DQ subflags имеют разумные значения
print(f'\n  DQ subflag distribution (this flight):')
for col in ['gap_flag', 'dq_derivative_bad', 'phase_inconsistent',
            'energy_deviation_extreme']:
    n_one = int((test_data[col] == 1).sum())
    print(f'    {col:>20s}: {n_one:,} points ({n_one / len(test_data) * 100:.1f}%)')

print(f'\nГотово')


=== Step 10: Sanity check (read highest-risk flight via filter) ===
  Picked flight 256711692 (max risk_score=1.0000)
  Read time: 0.66s
  Points: 7,050
  Columns (30): ['row_id', 'flight_id', 'timestamp', 'flight_phase', 'phase_name', 'split', 'latitude', 'longitude', 'altitude', 'groundspeed', 'vertical_rate', 'energy_ratio', 'energy_deviation', 'dq_hard_flag', 'dq_soft_flag', 'feature_quality_flag', 'is_clean_reference', 'dt_sec', 'gap_flag', 'dq_derivative_bad', 'phase_inconsistent', 'energy_deviation_extreme', 'if_score_phase_pct', 'hdb_score_phase_pct', 'lstm_score_max_phase_pct', 'lstm_score_count', 'ensemble_score', 'ensemble_count', 'risk_score', 'risk_level']
  Risk score range: [0.0127, 1.0000]
  Risk level distribution:
    low: 5,706
    medium: 1,023
    high: 321
    unknown: 0
  Phase distribution:
    cruise: 3,982
    climb: 1,197
    descent: 1,178
    approach: 610
    landing: 57
    takeoff: 26
    ground: 0

  DQ subflag distribution (this flight):
             

## 11. Summary

In [13]:
print('\n' + '=' * 60)
print('dashboard_points_v3.parquet PREPARED')
print('=' * 60)
print(f'\nFile: {DASHBOARD_POINTS_PATH}')
print(f'Size: {size_gb:.2f} GB')
print(f'Columns: {len(REQUIRED_OUTPUT_COLS)}')
print(f'  Identity: row_id, flight_id, timestamp, flight_phase, phase_name, split')
print(f'  Geo: latitude, longitude')
print(f'  Physics: altitude, groundspeed, vertical_rate')
print(f'  Energy: energy_ratio, energy_deviation')
print(f'  DQ aggregate: dq_hard_flag, dq_soft_flag, feature_quality_flag, is_clean_reference')
print(f'  DQ subflags: dt_sec, gap_flag, dq_derivative_bad, phase_inconsistent, energy_deviation_extreme')
print(f'  Model scores: if/hdb/lstm phase_pct + lstm_score_count')
print(f'  Ensemble: ensemble_score, ensemble_count')
print(f'  Risk: risk_score, risk_level')
print(f'\nDashboard can read via:')
print(f'  pq.read_table(path, filters=[("flight_id", "=", X)])')
print(f'\nReady for 04_dashboard.py rewrite.')


dashboard_points_v3.parquet PREPARED

File: /content/drive/MyDrive/thesis_processed/models_v3_artifacts/dashboard_points_v3.parquet
Size: 3.16 GB
Columns: 30
  Identity: row_id, flight_id, timestamp, flight_phase, phase_name, split
  Geo: latitude, longitude
  Physics: altitude, groundspeed, vertical_rate
  Energy: energy_ratio, energy_deviation
  DQ aggregate: dq_hard_flag, dq_soft_flag, feature_quality_flag, is_clean_reference
  DQ subflags: dt_sec, gap_flag, dq_derivative_bad, phase_inconsistent, energy_deviation_extreme
  Model scores: if/hdb/lstm phase_pct + lstm_score_count
  Ensemble: ensemble_score, ensemble_count
  Risk: risk_score, risk_level

Dashboard can read via:
  pq.read_table(path, filters=[("flight_id", "=", X)])

Ready for 04_dashboard.py rewrite.
